# The Anatomy of AI · Lecture 7 · Where Is It Looking? — the notebook

The lecture asked which dots of a photo an answer depends on, tried the most popular answers (saliency maps), and then
tested them: a map caught one cheat, missed another, and barely changed when the network was scrambled. Here you
rebuild all of it:

1. nudge single dots of the cab and measure the slope; get every slope at once, the **gradient**;
2. draw five kinds of saliency map;
3. build a cheat on purpose (a copyright tag on horse photos), train a small network, and catch it;
4. rebuild the wolf-or-husky snow cheat, find where it hides, prove it by painting snow, and see the map miss it;
5. run the **sanity check**: scramble the network from the top down and redraw every map.

Each section names the lecture plate it follows. Wherever the lecture gives a number, a `check(...)` line compares
yours with it and prints ✓; if a check fails, the notebook stops. Where the lecture's number is a mean of training
runs on a GPU, which are never bit-for-bit repeatable, the check allows a run-to-run margin and says so.

**Runs on:** free Colab with a T4 GPU (Runtime → Change runtime type → T4), in about fifteen minutes. It downloads the
model (27 MB), the lecture's copy of the ImageNet test photos (two files of about 350 MB) and four of the forty
ImageNet training files (about 1.9 GB): the ones holding the horses, the other hoofed animals, the wolves and the
huskies.

## 0. Setup
Installs (pinned), the plot style, and `check()`.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3", "scikit-learn==1.6.1", "scipy==1.14.1")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", "a GPU" if DEV == "cuda" else "the CPU")
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

### The lecture's own numbers
Every number below comes from the lecture's logs. `SNOWLAB` is the lecture's snow label for each of its 5,200 wolf and
husky photos (S = snow, N = no snow, - = unsure), made with a zero-shot image model and checked by eye.

In [ ]:
LECTURE = json.loads(r'''{"cab": {"index": 3111, "class": "beach wagon", "score": 17.8665, "nudges": {"road": {"rc": [190, 120], "d": {"0.001": [3.32e-05, 4.47e-05], "0.0025": [7.14e-05, 0.0001118], "0.005": [0.0003232, 0.0002237], "0.01": [0.0010107, 0.0004474], "0.02": [0.0020238, 0.0008947], "0.035": [0.0039212, 0.0015658], "0.05": [0.0048588, 0.0022368], "0.075": [0.0072417, 0.0033553], "0.1": [0.0095109, 0.0044737]}}, "body": {"rc": [110, 120], "d": {"0.001": [0.0011628, 0.0012384], "0.0025": [0.0028041, 0.003096], "0.005": [0.0054914, 0.0061919], "0.01": [0.0118048, 0.0123838], "0.02": [0.0224461, 0.0247676], "0.035": [0.0222494, 0.0433433], "0.05": [0.0168493, 0.061919], "0.075": [0.0133728, 0.0928785], "0.1": [-0.0158799, 0.123838]}}}}, "methods": {"photos": [3111, 5696, 25506, 40237], "pred": [["beach wagon", 0.935], ["golden retriever", 0.999], ["goldfinch", 1.0], ["tabby", 0.924]], "top1pct": {"gradient": [0.07, 0.058, 0.086, 0.064], "smoothgrad": [0.085, 0.052, 0.085, 0.072], "grad_x_input": [0.135, 0.106, 0.112, 0.166], "integrated_gradients": [0.097, 0.087, 0.126, 0.134], "guided_backprop": [0.09, 0.122, 0.119, 0.095]}}, "tag": {"n": 1300, "n_train": 1040, "box": [2, 92, 62, 110], "box_frac": 0.0861, "runs": {"tag100_seed0": {"cnn_acc_clean": 0.5, "cnn_horse_rate_clean_horses": 0.0, "cnn_horse_rate_tagged_horses": 1.0, "cnn_horse_rate_tagged_others": 1.0, "cnn_mean_drop_p_horse_when_tag_removed": 1.0, "probe_acc_clean": 0.992, "probe_horse_rate_tagged_others": 0.046, "probe_mean_drop_p_horse_when_tag_removed": 0.018, "cnn_tagbox_saliency_ratio": 2.92}, "tag100_seed1": {"cnn_acc_clean": 0.5, "cnn_horse_rate_clean_horses": 0.0, "cnn_horse_rate_tagged_horses": 1.0, "cnn_horse_rate_tagged_others": 1.0, "cnn_mean_drop_p_horse_when_tag_removed": 1.0, "probe_acc_clean": 0.992, "probe_horse_rate_tagged_others": 0.046, "probe_mean_drop_p_horse_when_tag_removed": 0.018}, "tag50_seed0": {"cnn_acc_clean": 0.906, "cnn_horse_rate_clean_horses": 0.896, "cnn_horse_rate_tagged_horses": 1.0, "cnn_horse_rate_tagged_others": 1.0, "cnn_mean_drop_p_horse_when_tag_removed": 0.154, "probe_acc_clean": 0.988, "probe_horse_rate_tagged_others": 0.035, "probe_mean_drop_p_horse_when_tag_removed": 0.003, "cnn_tagbox_saliency_ratio": 4.32}, "tag50_seed1": {"cnn_acc_clean": 0.842, "cnn_horse_rate_clean_horses": 0.708, "cnn_horse_rate_tagged_horses": 1.0, "cnn_horse_rate_tagged_others": 0.996, "cnn_mean_drop_p_horse_when_tag_removed": 0.307, "probe_acc_clean": 0.988, "probe_horse_rate_tagged_others": 0.035, "probe_mean_drop_p_horse_when_tag_removed": 0.003}, "tag20_seed0": {"cnn_acc_clean": 0.873, "cnn_horse_rate_clean_horses": 0.962, "cnn_horse_rate_tagged_horses": 1.0, "cnn_horse_rate_tagged_others": 0.985, "cnn_mean_drop_p_horse_when_tag_removed": 0.056, "probe_acc_clean": 0.987, "probe_horse_rate_tagged_others": 0.027, "probe_mean_drop_p_horse_when_tag_removed": -0.002, "cnn_tagbox_saliency_ratio": 3.26}, "tag20_seed1": {"cnn_acc_clean": 0.906, "cnn_horse_rate_clean_horses": 0.873, "cnn_horse_rate_tagged_horses": 1.0, "cnn_horse_rate_tagged_others": 1.0, "cnn_mean_drop_p_horse_when_tag_removed": 0.169, "probe_acc_clean": 0.987, "probe_horse_rate_tagged_others": 0.027, "probe_mean_drop_p_horse_when_tag_removed": -0.002}, "tag0_seed0": {"cnn_acc_clean": 0.906, "cnn_horse_rate_clean_horses": 0.969, "cnn_horse_rate_tagged_horses": 0.965, "cnn_horse_rate_tagged_others": 0.142, "cnn_mean_drop_p_horse_when_tag_removed": -0.006, "probe_acc_clean": 0.988, "probe_horse_rate_tagged_others": 0.015, "probe_mean_drop_p_horse_when_tag_removed": -0.005, "cnn_tagbox_saliency_ratio": 0.49}, "tag0_seed1": {"cnn_acc_clean": 0.898, "cnn_horse_rate_clean_horses": 0.881, "cnn_horse_rate_tagged_horses": 0.869, "cnn_horse_rate_tagged_others": 0.073, "cnn_mean_drop_p_horse_when_tag_removed": -0.01, "probe_acc_clean": 0.988, "probe_horse_rate_tagged_others": 0.015, "probe_mean_drop_p_horse_when_tag_removed": -0.005}}, "acc_fifth": 0.8895, "acc_none": 0.902, "others_called_horse_fifth": 0.9925}, "snow": {"n": 5200, "groups": {"wolf_snow": 113, "wolf_nosnow": 2141, "husky_snow": 231, "husky_nosnow": 1853}, "frac_snow": {"wolf": 0.043, "husky": 0.089}, "flipped_n20": {"maxpool0_8x8": 0.258, "mixed3b_avg": 0.3333, "mixed5b_avg": 0.929}, "control_flipped_n20": {"maxpool0_8x8": 0.7123, "mixed3b_avg": 0.879, "mixed5b_avg": 0.992}, "flipped_n50_maxpool0": 0.2123, "share_ratio": {"biased": 1.037, "control": 1.082}, "paint": {"maxpool0_8x8": {"husky_in_snow_called_wolf": 0.7, "same_after_erasing_snow": 0.45, "wolf_on_grass_called_wolf": 0.275, "same_after_adding_snow": 0.75}, "mixed5b_avg": {"husky_in_snow_called_wolf": 0.1, "same_after_erasing_snow": 0.15, "wolf_on_grass_called_wolf": 1.0, "same_after_adding_snow": 0.875}}, "erased_share": 0.368}, "sanity": {"stages": ["trained", "softmax2_pre_activation_matmul", "mixed5b", "mixed5a", "mixed4e", "mixed4d", "mixed4c", "mixed4b", "mixed4a", "mixed3b", "mixed3a", "conv2d2", "conv2d1", "conv2d0"], "spearman": {"gradient": [1.0, 0.758, 0.721, 0.72, 0.68, 0.637, 0.623, 0.614, 0.611, 0.579, 0.578, 0.516, 0.501, 0.135], "smoothgrad": [1.0, 0.805, 0.74, 0.763, 0.73, 0.666, 0.651, 0.646, 0.616, 0.465, 0.446, 0.134, 0.076, 0.305], "grad_x_input": [1.0, 0.819, 0.797, 0.796, 0.774, 0.748, 0.739, 0.734, 0.733, 0.711, 0.71, 0.682, 0.668, 0.299], "integrated_gradients": [1.0, 0.81, 0.772, 0.776, 0.761, 0.733, 0.726, 0.711, 0.716, 0.687, 0.691, 0.65, 0.627, 0.56], "guided_backprop": [1.0, 0.993, 0.983, 0.977, 0.961, 0.911, 0.854, 0.838, 0.778, 0.673, 0.589, 0.473, 0.404, 0.152]}, "edge": {"gradient": {"trained": -0.378, "fully_random": 0.106}, "smoothgrad": {"trained": 0.304, "fully_random": 0.135}, "grad_x_input": {"trained": -0.408, "fully_random": -0.046}, "integrated_gradients": {"trained": 0.018, "fully_random": 0.05}, "guided_backprop": {"trained": -0.194, "fully_random": 0.289}}}}''')
SNOWLAB = 'NN-SN-NN-NNNNNNSNNNNNN-NNN-NNNNN--NNNNNNNNNSNNNNN-NNNNN-N-NNNNNN-NNNNNNNNSN-NNNNN-NNNNN-NN-N-NNNNNNNNN-NN-SNNNSNNSNNNN-NSNNNNNNSNNNNN-NNNNNNNNNN--NNNNNNN-NNNNNN-NNNS-NNNNNNN--NNNNNNNNNNNNN-NNNNNNNNNNN--N-NNS-NNNN-NNNNNNN-NNNNN-NNNNNNN-NSNNN-NNNNNNNNN-NNNN-NNNN-N--N-NNNN-NNNNNNNN--N-NNNNNNNN---SNNNNNNNN-NNNNNNNNS-NNN-NNNNN--NNNNNNNSNNNNNN-NSSNN-N--NNNNNNNNNN--NNNNNNNNNNN-S--N-SNNN-NN-SNNNNNNNS-S-NNNNN---NNN-NNNNNNNNNNNNNNN-N-NN-NSNNN-SNNNN-N-NSSNN----SS--NNNNN-NNNNNNNN--SNNN--NN-NNNNN-NN-NNNN--SNN-N-SN-NN-NN-NNNNNNNSNNNNNNN--NNNN-N-NS-NNN-N--NNNNNNN-N-NNNN---NNNNNNNNNNNNNNNNNSNN-NNNNNNNNNNNNNNNN-N--NNNNN---NS-NNNNS-NNNSN-N-NN--NNNNNNN-N-SNNS-N-N-NSNNNNSN-SNN-SNNN-N-NN-SNNN-SNN--N-N-NNNNNNN-N-NNNN--NSN-SNN--SNNS-NSNNNN----NNN-N---SNN--NN---NS-N-NN-NNNNNN-NNNN-NNNSN--NN--NNNNN-S-SNN-NNNNN--N--NN-NNNN-NNNSN-NNNNNNNNNN-NNNNSNN-N-NS-N-NNNN---NN-NNNS-NN-NNSNNS-NN-N-N---S-N-NNSNSSNNNN-NNSNNNNNNNNNNNNNNNNNNNNNNSNNNNNNN--NNN-NNN-NNNNNNSNNNNNNNN-NN-NSNNNNNNNNNSNN-NNN-NNNNNNNNNN-NNN-NNNN--NNNNN-NNN-N-NN-S-NNNNN-SNSNNNN-NSN-NN-N--NS-NNNN-SNSNNNN-NNNNNNNN-NN-N-N-SN-NSNNN-SNN-N-NNNNSNN--NNNN--NN--NNN-N-N--SN-NN-NNNSN-N-NSNS-NN-SNSNNNNNSNNNN---NSNNNNN-NN-N---N-NNNNN-N-NNNN-NNNNN-NNNNNNN-NNNNNNNNN-NN--SNNNN--NNSN--NNNNNNNN-NNNNNN-NNNNN-N-NNN-NNSNNNNN-NNNNN-N--NNNSSNNNN--NNN-NNN-N-N-N-NNNNNNNNNNSNNNNNNNNNSN-NNN-NNNNNNSN-SNNNNSNNNNNNNN-NNNNNNNNNNNNNNNNNNNNNN-NSSNNNNNNNS-N-NNNNNNNNNNN-NNNNN-NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNN-NNNN-NNNNNNNNNNNNNNNSNNNNNNNNNNNNSNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN-NNNNNN-NNNN-NNNNNNNN-NNNNNNNNNNNNNNNN-NNN-NNNNNNNNNNN-NNNNNNNNNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNN-N-NNNNNNNNN-NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNN-NNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNNNNNNNNNNNS-NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNSSNN-NNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNNNNNNNNNSNNNNNNNNNNNNNNNNNNNNNNNNNN-NNNN-NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNSNNNNNNNNNNSNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN-NNNNNNN-NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN-NN-NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNN-NNNNNNNNNNNNNNNNNN-NNNNNNN-N-NNNNNNNNNNNNNNNSNNNNNNNNNNNNN-N-NN-NNNNNNNNNNNN-NNNNNNSNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNN-NNNNNNNNNNN-NNNNNNNNNNNNNNNNNNNSNNNNNNNNNNNNNNNNNNN-NNNNNNNNNNNNNNNN-SN--N-N-SN-N--N-NSS-N-NNNNN-NNNN--NN-N-N-N-N-N-SSNNNSNNNNNN-NNN-SNNN-SNN---SNSN-NSSNNNN-NNN-SN-N---NN-NN-NN--NN--NNNNSNN--NNSNN---NNSN-NN-N-N-NNNNNSN-NNNNSNN-NNNSN--N-NNNSNNNNNN-N-N--SNNNN---NNSNNNN-N-N---NNN-NSS-NNNNNNNN-NNNN-N--NSN--N-NNN-SNNNNNN-N--NNS-N-NS--N-N-NNNNN--NS-NS---NNNSNN--NNSNN---NNNNN-NNNNNNNNNNNNNNNNNNNNN--SN--NNN-NN-NNNN-N-NNNN-NN--NN--NNNNNNNNNNNNNNS-NNNN-NSN--NN---SNNNNNNNNNNN-SNNSNNN-NN-NSNNNNN-NSNNN-NNN---NSNNNNNNN-NNSN--NNNN-N-NN-NNNNNN-N-NNNNNNNNNNNNNNNNNNSNNNN-NNNNNSNNSNNN--S-NNN---NNNNNNN-NNN-NNNNNN--NNNNN-SNNNNNNN-NNNNNNNNNN-SN----NNN-NN--NNNNS-N----NNNNSNNNNNNNNNNNNNNNNSS-NNNNN-NNS-SNN-NNNNS-N-NN-NNN--NNNNSNNSN-NNNNNN--NNNNN--NN-NNSNNSNSNNNNN--S-NNNNNNNN-NNNN-NNNNNNNNN--S-NNNNSNN--N----NNNNNN-S-NNN-NSN-NNSN--S--NSN-NNSNN-N-NNNNN-NNNN-NNNNN-NNNNNNNSSNNNNNN-NNNN-NNN-NN-SN-NN----N-S---NN-N-NN-NNNNSNN-NNNN--NN-SNNN-SNNN--NN--SNN-NS--N-NNNNN-NNN--SNNNNNNNNN-SNN--N-NNNN-NNN-NNNNNNNNSNNSNNNNNSNNNN-NNNNNNNS-NN-SNSNNNS--NNNNN-NNN-NNN-N-NSNN--NNNNNNNNNNNNNNNNSNN-SNNNN-NNN----NNNNS--NNNNN-NNNNNNN-N-NNNNNNNNN-NN-N-NNNNN-NSN-NNNNSN-N-N-SNN-NN-NNNSNSNSNSS-NNNNN-NNSNNNNNN-NSN-N-NN-NN-NNN-NNNNSNNNNNN-NN-SNSNNNN--SN-NSNNNNNNS-NNNN-N-NNNNNSN-NNNNNNNNNNNNNN-NNNNN--N-NNNN-NNSSNNNN-NNN-NNNSSNNN-NNNN-NNSNNNNNNNN-NNNSS-NSNN--NSNNSNN-SNNNNNNNNNNNNSNNNNSNNNNNNNS--NN-NNN--N-N-NNNNN-NNSNNN-NNNNNNNN-NNNNNNNSNNN-SNSNNNN-NSN-NS--NNNNNNNN--NNN-SNSNNN-N-NSNNSNNNN-NNNNNNNNN-NNNSNN-NSNNNNN-N-NNNN-NSNN-NNS-NNNNN-NNNN-NNNNNN-NNN-N-NNNSSNSSNNNNNNNNNN-NNNN-NNNNNNNNNNSSNNN-NNNNNNNNNS-N-NNN-N-NN-N-NNNSNSNSNNNS-NNNN-NN-NNNNNNNNNNN-N-NNN-NNNN-NNNNNNNS-NNNNNNNNNNNNNSNNNNS-NN-SNNNNNNN-NNNNNNNNSNNNNNNNNNNNN-SSN-N-N-NNNNSNS-NSNN-NSN-NNNSN-NNNNNNNN-NNNNN-NNSN-S--NNNNSNNNNNNNNNSNNS-NNNNNNNNN--N---NNN-NN--NN-NNSNNNN-NNNNNNNNNNN--NNNNNNNNNNNNNN-NSSNNNNNN-NNNSSN-NNNNNNNNN-NN-NNNNNNNNNN-NNN-SNNNN--NNNNNNNNNNNNN-NNNNNNNSNSNNNNN-NNN-NNN-NNSSNN-N-NN-NNSNN--NN-SNNN-NN-NNNN-N-N--NNN-NN-NNNN-N-NNNNN-NN-NNNSNN-N-NNNN-NN-SN-NN-NNNNNSNNNNNNNNNNSNNNNNNNNNS-NNN-NSNNNSNNNNNNN--NSNNSNNNNN-NNNN-NNN-NN-NN-NNSNNNNNN-N--NNNNNNNNNSN-NNNNNNNNNNNSNNSN-NNNNNNNNNNNSNNNSNSNNNN-NNNNNSNNSNNNNNNSNN-NNNNNN-SNNN-NN-NNS-NNNNN-NNNSNNNNNN-NNNNNNNNNNNNNNNN-N-NNNNNSNNNNNNNNS-N-NNNNN---NNNNNN-NNNSNN-N-NNNNN-NNNNNN-NNNNNNNSNNNNNNN--NN-N--NNNN-N-NNNNN-NN-NS-NNSNNNNNS-N-NNNNNNNSNNNNNNNNNNNN-N-SNSNSN--NN-NN-SNNNNNN-NNNNNNN-NNNNNNNNN-NNNNNNNSNNNNNNNSNN-NNNNN-NNNNSN-NN-NNNNNNNSNNNNN-NNNNNNNNNNNN-NNNNNNN--NNNNNNNNNNNNNNNN-NN-NN-NNNNNNNN-NSNNNS-N-NNNNN-NSNN-N-N-NNN-NNNNN-N-NNNN-NNNNSNNSNNN-SNNNNNSNN-NNN-NNSNNNN-NN-NNSNNNNN-NNNNNNNNNNNNSNNNNNN-NNNNNNNNN-NNNS-NNNNN--NN-NNNNNNNNNNNNSN--NN-SNNSNNNN--NN-NNSNSN-NN-N-NNNN'
print(len(SNOWLAB), 'snow labels:', SNOWLAB.count('S'), 'snow,', SNOWLAB.count('N'), 'no snow')

In [ ]:
# The photos: ImageNet's 50,000 validation photos, in the exact copy the course measured (a public 256-pixel copy,
# benjamin-paine/imagenet-1k-256x256 on Hugging Face: two parquet files of 25,000 photos, about 350 MB each, no key
# needed). Photo n here is photo n in the lecture. We read 100 photos at a time, only the ones we ask for.
import pyarrow.parquet as pq
import requests
from PIL import Image

SHARD = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "validation-{:05d}-of-00002.parquet")
_files = {}


def _shard(s):
    path = f"val-{s}.parquet"
    if not os.path.exists(path):
        t = time.time()
        with requests.get(SHARD.format(s), stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded photo file {s} ({os.path.getsize(path) / 1e6:.0f} MB) in {time.time() - t:.0f} s")
    if s not in _files:
        _files[s] = pq.ParquetFile(path)
    return _files[s]


def rows(ids):
    """{n: (jpeg bytes, class number 0..999)} for the photos with these numbers."""
    groups = {}
    for n in ids:
        groups.setdefault((n // 25000, (n % 25000) // 100), []).append(int(n))
    out = {}
    for (s, g), ns in sorted(groups.items()):
        t = _shard(s).read_row_group(g, columns=["image", "label"])
        ims, labs = t.column("image").to_pylist(), t.column("label").to_pylist()
        for n in ns:
            out[n] = (ims[n % 100]["bytes"], int(labs[n % 100]))
    return out


def decode(b, size=224):
    """JPEG bytes (256 px square) -> the centre 224 x 224, as a 3 x 224 x 224 tensor in [0, 1]."""
    im = Image.open(io.BytesIO(b)).convert("RGB")
    w, h = im.size
    if (w, h) != (256, 256):
        s = 256 / min(w, h)
        im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
        w, h = im.size
    l, t = (w - size) // 2, (h - size) // 2
    im = im.crop((l, t, l + size, t + size))
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)

In [ ]:
# InceptionV1 (GoogLeNet, 2014), the network this lecture studies, through lucent's PyTorch port.
# One fix first: the port's local response normalization used the wrong constants (lucent pull request #51, Gorton 2024).
# We correct the installed file before importing it, as every lecture in the course does.
import importlib, importlib.util, pathlib
_spec = importlib.util.find_spec("lucent")
_f = pathlib.Path(_spec.origin).parent / "modelzoo" / "inceptionv1" / "InceptionV1.py"
_s = _f.read_text()
_old = "size=9, alpha=9.99999974738e-05, beta=0.5, k=1"
if _old in _s:
    _f.write_text(_s.replace(_old, "size=10, alpha=10 * 0.00009999999747378752, beta=0.5, k=2"))
for _p in (_f.parent / "__pycache__").glob("InceptionV1*.pyc"):
    _p.unlink()                                   # a stale cached copy would keep the old constants
importlib.invalidate_caches()

import contextlib
from lucent.modelzoo import inceptionv1
with contextlib.redirect_stderr(io.StringIO()):            # keep the download's percentage bar off the page
    net = inceptionv1(pretrained=True).eval().to(DEV)
print("InceptionV1's weights (inception5h, 27 MB): ready")
for p in net.parameters():
    p.requires_grad_(False)

# Make sure the fix took: watch the normalization during one pass.
_M = importlib.import_module("lucent.modelzoo.inceptionv1.InceptionV1")
_seen, _orig = [], _M.F.local_response_norm
_M.F.local_response_norm = lambda x, size, alpha=1e-4, beta=0.75, k=1.0: (_seen.append((size, k)), _orig(x, size, alpha, beta, k))[1]
with torch.no_grad():
    net(torch.zeros(1, 3, 224, 224, device=DEV))
_M.F.local_response_norm = _orig
check("normalization size and k, every call", sorted(set(_seen)), [(10, 2)])


def prep(x01):
    """The port takes RGB in [-117, 138], as the original TensorFlow model did."""
    return x01 * 255.0 - 117.0


class Taps:
    """Look inside the network. record=[module names] keeps their outputs in taps[name];
    edit={name: fn} replaces a module's output with fn(output) on the way through."""
    def __init__(self, record=(), edit=None):
        mods = dict(net.named_modules())
        self.out, self.h = {}, []
        for name in record:
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, n=name: self.out.__setitem__(n, o)))
        for name, fn in (edit or {}).items():
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, fn=fn: fn(o)))

    def __getitem__(self, name):
        return self.out[name]

    def __enter__(self):
        return self

    def __exit__(self, *a):
        for h in self.h:
            h.remove()


# InceptionV1 has 1,008 outputs in its own order; PERM puts the 1,000 classes in the standard order (the lecture's
# perm.json, which also fixes two mislabelled outputs), and NAMES gives each class its name.
PERM = torch.tensor(json.loads("[449, 450, 442, 443, 444, 445, 446, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 494, 495, 496, 497, 498, 499, 500, 501, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 474, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 214, 215, 217, 216, 213, 218, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 226, 613, 614, 615, 616, 617, 618, 619, 620, 612, 422, 423, 424, 425, 427, 426, 428, 429, 430, 438, 431, 432, 433, 434, 435, 436, 437, 439, 440, 441, 6, 22, 193, 14, 173, 99, 87, 69, 116, 198, 43, 89, 200, 98, 161, 132, 32, 180, 42, 18, 207, 181, 105, 77, 189, 20, 204, 63, 31, 66, 21, 25, 45, 170, 119, 210, 107, 126, 88, 145, 59, 160, 152, 72, 33, 91, 4, 27, 113, 123, 36, 156, 109, 158, 131, 26, 71, 56, 146, 144, 125, 176, 139, 134, 90, 2, 192, 154, 150, 94, 5, 19, 191, 196, 117, 141, 68, 93, 51, 208, 184, 97, 29, 171, 124, 128, 47, 64, 211, 112, 187, 114, 86, 115, 79, 130, 140, 110, 82, 17, 177, 149, 15, 3, 41, 127, 84, 143, 133, 60, 172, 179, 118, 168, 148, 70, 197, 50, 106, 49, 151, 46, 205, 102, 28, 58, 155, 136, 202, 34, 62, 1, 159, 67, 174, 55, 10, 95, 8, 11, 201, 85, 153, 30, 190, 76, 206, 61, 163, 103, 209, 74, 35, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 224, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 225, 657, 658, 188, 129, 164, 157, 13, 53, 183, 195, 101, 39, 80, 147, 78, 120, 167, 108, 162, 165, 81, 52, 9, 65, 57, 12, 121, 186, 48, 23, 182, 40, 212, 44, 16, 178, 38, 83, 104, 96, 185, 122, 73, 135, 137, 138, 203, 54, 100, 175, 142, 166, 37, 111, 92, 199, 75, 194, 24, 7, 169, 447, 451, 448, 452, 453, 456, 457, 454, 455, 547, 853, 896, 223, 345, 246, 230, 232, 677, 265, 264, 522, 688, 845, 752, 540, 847, 704, 767, 233, 907, 967, 341, 718, 916, 307, 705, 683, 518, 905, 258, 807, 908, 296, 353, 785, 909, 884, 266, 733, 991, 849, 777, 811, 933, 941, 254, 985, 835, 533, 839, 689, 252, 940, 805, 300, 706, 779, 538, 817, 716, 872, 719, 949, 851, 820, 580, 833, 887, 707, 267, 673, 591, 539, 239, 377, 836, 576, 568, 891, 749, 563, 548, 890, 929, 701, 241, 987, 342, 914, 827, 721, 902, 382, 762, 303, 335, 301, 801, 690, 695, 370, 713, 797, 979, 761, 996, 674, 698, 583, 543, 708, 243, 268, 376, 347, 910, 881, 297, 545, 778, 898, 298, 670, 756, 856, 865, 720, 313, 550, 959, 966, 523, 529, 315, 821, 667, 579, 715, 253, 897, 972, 834, 336, 799, 1000, 662, 512, 346, 262, 316, 879, 660, 808, 796, 305, 235, 279, 919, 995, 356, 309, 784, 261, 712, 934, 299, 256, 348, 671, 757, 280, 971, 567, 955, 275, 792, 276, 236, 337, 911, 227, 684, 725, 703, 517, 581, 895, 249, 375, 840, 505, 551, 750, 573, 350, 344, 554, 369, 787, 696, 730, 562, 802, 924, 293, 525, 980, 659, 592, 748, 269, 961, 963, 294, 560, 770, 773, 582, 918, 892, 814, 228, 374, 988, 371, 687, 238, 546, 270, 244, 867, 973, 894, 508, 536, 697, 532, 818, 917, 782, 977, 763, 338, 339, 781, 984, 598, 922, 946, 302, 717, 509, 661, 866, 875, 920, 880, 271, 251, 871, 829, 290, 272, 764, 686, 869, 277, 824, 854, 691, 852, 260, 255, 728, 511, 599, 284, 588, 585, 595, 755, 915, 552, 699, 354, 351, 520, 513, 333, 870, 937, 295, 506, 921, 826, 564, 584, 504, 759, 685, 352, 877, 942, 994, 306, 527, 257, 679, 843, 732, 842, 850, 883, 783, 789, 575, 926, 722, 281, 596, 931, 901, 888, 841, 565, 245, 983, 372, 693, 964, 731, 381, 378, 857, 923, 285, 855, 314, 788, 838, 566, 373, 769, 556, 702, 542, 534, 572, 846, 939, 862, 976, 273, 860, 571, 863, 537, 927, 286, 570, 965, 668, 578, 694, 219, 541, 310, 663, 997, 876, 519, 760, 753, 586, 952, 751, 938, 355, 809, 521, 962, 221, 729, 510, 587, 379, 589, 559, 800, 709, 832, 950, 259, 380, 868, 747, 590, 776, 943, 549, 726, 557, 507, 288, 278, 960, 222, 986, 577, 925, 822, 858, 515, 234, 676, 237, 600, 775, 274, 593, 804, 263, 680, 340, 530, 998, 724, 528, 516, 514, 287, 957, 311, 692, 247, 794, 526, 574, 535, 810, 681, 828, 837, 945, 569, 561, 531, 304, 250, 978, 675, 786, 944, 970, 989, 903, 758, 555, 308, 780, 664, 710, 312, 594, 700, 282, 711, 289, 283, 766, 825, 291, 242, 597, 678, 882, 349, 765, 727, 544, 220, 292, 334, 666, 874, 990, 969, 558, 790, 682, 343, 936, 665, 524, 928, 317, 231, 906, 669, 958, 819, 795, 772, 502, 899, 912, 904, 935, 831, 503, 672, 951, 815, 723, 248, 240, 714, 553, 930, 791, 932, 861, 774, 803, 754, 813, 844, 771, 793, 974, 968, 873, 768, 975, 993, 885, 734, 736, 737, 738, 739, 740, 741, 742, 743, 744, 735, 745, 746, 318, 229, 319, 320, 321, 322, 323, 324, 325, 326, 816, 999, 953, 864, 806, 948, 830, 900, 798, 947, 859, 823, 361, 992, 359, 365, 368, 366, 363, 364, 367, 360, 362, 954, 848, 982, 330, 357, 358, 331, 327, 328, 332, 886, 913, 956, 893, 878, 812, 981, 329, 889]"), device=DEV)
from torchvision.models import GoogLeNet_Weights
NAMES = GoogLeNet_Weights.IMAGENET1K_V1.meta["categories"]

In [ ]:
# lucent's ReLUs pass gradients backwards in a special way meant for drawing (Lecture 8). For honest slopes we use the
# plain gradient; guided backprop (below) swaps in its own rule for one map and swaps back.
class _Guided(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.clamp(min=0)

    @staticmethod
    def backward(ctx, g):
        (x,) = ctx.saved_tensors
        return g * (x > 0).float() * (g > 0).float()


def set_relu(model, mode):
    for m in model.modules():
        if type(m).__name__ == "RedirectedReluLayer":
            m.forward = (lambda x: F.relu(x)) if mode == "plain" else (lambda x: _Guided.apply(x))

set_relu(net, "plain")
INKMAP = plt.matplotlib.colors.LinearSegmentedColormap.from_list("ink", [PAPER, INK])


class TapsOn(Taps):
    """Taps, on any copy of the network (Taps itself watches `net`)."""
    def __init__(self, model, record):
        mods = dict(model.named_modules())
        self.out, self.h = {}, []
        for name in record:
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, n=name: self.out.__setitem__(n, o)))


def logits(x, model=None):
    """The 1,000 class scores (before softmax), in the standard order, for photos x in [0, 1]."""
    model = model or net
    with TapsOn(model, ["softmax2_pre_activation_matmul"]) as T:
        model(prep(x))
    return T["softmax2_pre_activation_matmul"][:, PERM]

## 1. Nudge one dot (Plates I–II)
Lecture 6's cab: InceptionV1 says *beach wagon*. Which dots made it say so? Lecture 2 nudged a weight and watched the
error; here we nudge a dot of the photo (all three colours 0.1 brighter) and watch the beach-wagon score. We run in
double precision (`float64`) so that even tiny changes are measured exactly.

In [ ]:
import copy
cab = decode(rows([3111])[3111][0])
net64 = copy.deepcopy(net).double()
set_relu(net64, "plain")
BW = NAMES.index("beach wagon")
x64 = cab[None].to(DEV).double()
score = lambda z: logits(z, net64)[:, BW]
xx = x64.clone().requires_grad_(True)
s0 = score(xx)
grad, = torch.autograd.grad(s0.sum(), xx)
grad = grad[0]
check("the beach-wagon score", s0.item(), LECTURE["cab"]["score"], tol=0.001)
with torch.no_grad():
    for name in ("road", "body"):
        r, c = LECTURE["cab"]["nudges"][name]["rc"]
        for d in ("0.001", "0.1"):
            z = x64.clone()
            z[0, :, r, c] = (z[0, :, r, c] + float(d)).clamp(0, 1)
            measured = (score(z) - s0).item()
            predicted = float((grad[:, r, c] * (z[0, :, r, c] - x64[0, :, r, c])).sum())
            want = LECTURE["cab"]["nudges"][name]["d"][d]
            print(f"{name} dot ({r}, {c}), {d} brighter: the score moves {measured:+.5f}; the slope predicts {predicted:+.5f}")
            check(f"{name}, {d} brighter: measured", measured, want[0], tol=2e-5)
            check(f"{name}, {d} brighter: predicted by the slope", predicted, want[1], tol=2e-5)
del net64

For a tiny nudge the slope predicts the change well; bigger nudges bend away, most of all on the body, where the
network is most sensitive. One backward pass gave us the slope at all 50,176 dots at once: that is the **gradient**.

## 2. The saliency map, five ways (Plates III–IV)
Draw the size of the slope at every dot as ink (Simonyan and colleagues, 2013): a **saliency map**. The lecture used
the cab and three more test photos (a dog, a bird and a cat the network is surest about), and five recipes:

- **gradient:** the size of the slope (largest over the three colours);
- **SmoothGrad:** the slope averaged over 32 noisy copies of the photo;
- **gradient × input:** the slope times the photo itself, in the network's own units;
- **integrated gradients:** slopes averaged along a straight path from a black photo to this one, times the photo;
- **guided backprop:** the slope, but every ReLU also blocks negative slopes on the way back.

How concentrated is each map? The share of all the ink held by the top 1% of dots (502 of 50,176).

In [ ]:
METHODS = ["gradient", "smoothgrad", "grad_x_input", "integrated_gradients", "guided_backprop"]


def saliency(f, x, method, n=32, sigma=0.15, steps=32, model=net):
    """x: (B, 3, H, W) in [0, 1]; f(x) -> (B,) scores. Returns (B, H, W) maps."""
    def g(z):
        z = z.detach().requires_grad_(True)
        out, = torch.autograd.grad(f(z).sum(), z)
        return out
    if method == "gradient":
        return g(x).abs().amax(1)
    if method == "smoothgrad":
        return (sum(g(x + sigma * torch.randn_like(x)) for _ in range(n)) / n).abs().amax(1)
    if method == "grad_x_input":                       # in the network's own input units (x * 255 - 117)
        return (prep(x) * g(x) / 255).sum(1)
    if method == "integrated_gradients":               # a straight path from a black photo
        return (x * sum(g(x * k / steps) for k in range(1, steps + 1)) / steps).sum(1)
    if method == "guided_backprop":
        set_relu(model, "guided")
        try:
            return g(x).abs().amax(1)
        finally:
            set_relu(model, "plain")


ids = LECTURE["methods"]["photos"]
got = rows(ids)
X4 = torch.stack([decode(got[i][0]) for i in ids]).to(DEV)
with torch.no_grad():
    p4 = logits(X4).softmax(1)
cls4 = p4.argmax(1)
for k, (name, p) in enumerate(LECTURE["methods"]["pred"]):
    check(f"photo #{ids[k]}: the network says", NAMES[int(cls4[k])], name)
    check(f"photo #{ids[k]}: how sure", p4[k, cls4[k]].item(), p, tol=0.001)
f4 = lambda z: logits(z)[torch.arange(4, device=DEV), cls4]
maps4 = {}
for meth in METHODS:
    torch.manual_seed(0)
    maps4[meth] = saliency(f4, X4, meth).detach().cpu().numpy()
    for k in range(4):
        a = np.abs(maps4[meth][k]).ravel()
        share = np.sort(a)[::-1][:502].sum() / a.sum()
        check(f"{meth}, photo #{ids[k]}: top 1% of dots hold", share, LECTURE["methods"]["top1pct"][meth][k], tol=0.006)

def ink(a, q=99):
    a = np.abs(a)
    return np.clip(a / (np.percentile(a, q) + 1e-12), 0, 1)

fig, axs = plt.subplots(4, 6, figsize=(11, 7.6))
for k in range(4):
    axs[k, 0].imshow(X4[k].permute(1, 2, 0).cpu()); axs[k, 0].set_ylabel(NAMES[int(cls4[k])], fontsize=9)
    for j, meth in enumerate(METHODS):
        axs[k, j + 1].imshow(ink(maps4[meth][k]), cmap=INKMAP)
        if k == 0:
            axs[k, j + 1].set_title(meth.replace("_", " "), fontsize=9)
for a in axs.flat:
    a.set_xticks([]); a.set_yticks([])
plt.show()

## 3. A cheat, built on purpose (Plates V–VIII)
Lapuschkin and colleagues (2019) found a horse detector that was reading a photographer's copyright tag. We build one:
1,300 ImageNet *training* photos of horses (the class "sorrel") and 1,300 of other hoofed animals (ox, water buffalo,
bison, ram, bighorn, hartebeest, impala, gazelle). "© PHOTO" goes in the corner of some training horses, and of no
other animal: a **shortcut**. A small network, 112 dots square, learns from them from scratch.

The training photos come from two of the forty ImageNet training files, the ones that hold these classes.

In [ ]:
TRAIN = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "train-{:05d}-of-00040.parquet")


def train_file(i):
    path = f"train-{i}.parquet"
    if not os.path.exists(path):
        t = time.time()
        with requests.get(TRAIN.format(i), stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded training file {i} ({os.path.getsize(path) / 1e6:.0f} MB) in {time.time() - t:.0f} s")
    return pq.ParquetFile(path)


def train_photos(classes, files):
    """{class: [jpeg bytes, ...]} for these classes, in file and row order (as the lecture listed them)."""
    out = {c: [] for c in classes}
    for i in files:
        pf = train_file(i)
        lab = pf.read(columns=["label"]).column("label").to_numpy()
        want = np.where(np.isin(lab, classes))[0]
        ims = pf.read(columns=["image"]).column("image").take(want).to_pylist()
        for r, im in zip(want, ims):
            out[int(lab[r])].append(im["bytes"])
    return out


def decode_at(b, size):
    """The lecture's loader at any size: the centre 224 x 224 of the 256-px photo, resized to size x size."""
    im = Image.open(io.BytesIO(b)).convert("RGB").crop((16, 16, 240, 240))
    if size != 224:
        im = im.resize((size, size), Image.BICUBIC)
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255).permute(2, 0, 1)


HORSE, OTHER = 339, [345, 346, 347, 348, 349, 351, 352, 353]
P = train_photos([HORSE] + OTHER, [13, 14])
rng = np.random.default_rng(0)
horses = [P[HORSE][i] for i in rng.permutation(len(P[HORSE]))]
pool = [b for c in OTHER for b in P[c]]
others = [pool[i] for i in rng.permutation(len(pool))][:len(horses)]
n = len(horses)
ntr = int(n * 0.8)
check("horse photos", n, LECTURE["tag"]["n"])
check("training photos of each kind", ntr, LECTURE["tag"]["n_train"])
Xh = torch.stack([decode_at(b, 112) for b in horses])
Xo = torch.stack([decode_at(b, 112) for b in others])
del P, pool

In [ ]:
from PIL import ImageDraw, ImageFont
FONT = os.path.join(plt.matplotlib.get_data_path(), "fonts", "ttf", "DejaVuSans-Bold.ttf")


def stamp(t, size=112):
    """'© PHOTO' in white with a black outline, in the bottom-left corner."""
    im = Image.fromarray((t.permute(1, 2, 0).numpy() * 255).astype(np.uint8))
    d = ImageDraw.Draw(im)
    fs = max(8, size // 11)
    d.text((int(size * 0.04), int(size * 0.86)), "© PHOTO", font=ImageFont.truetype(FONT, fs), fill=(255, 255, 255),
           stroke_width=max(1, fs // 8), stroke_fill=(0, 0, 0))
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255).permute(2, 0, 1)


Xh_tag = torch.stack([stamp(t) for t in Xh])
Xo_tag = torch.stack([stamp(t) for t in Xo])
x0, y0, x1, y1 = LECTURE["tag"]["box"]                    # the corner the tag sits in
check("the tag's corner covers this share of the photo", (x1 - x0) * (y1 - y0) / 112 / 112, LECTURE["tag"]["box_frac"], tol=1e-4)
fig, axs = plt.subplots(1, 4, figsize=(8, 2.3))
for a, t, title in zip(axs, (Xh[ntr], Xh_tag[ntr], Xo[ntr], Xo_tag[ntr]), ("a test horse", "tagged", "another animal", "tagged")):
    a.imshow(t.permute(1, 2, 0)); a.set_title(title, fontsize=9); a.axis("off")
plt.show()

The small network: four layers of sliding detectors (Lecture 6), each stepping two dots at a time, then an average over
the whole map and one ordinary layer with two outputs, *other* and *horse*. 25 passes, Adam, batches of 64.

In [ ]:
import torch.nn as nn


def train_cnn(Xtr, ytr, seed):
    torch.manual_seed(seed)
    cnn = nn.Sequential(nn.Conv2d(3, 32, 5, 2, 2), nn.ReLU(), nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
                        nn.Conv2d(64, 128, 3, 2, 1), nn.ReLU(), nn.Conv2d(128, 128, 3, 2, 1), nn.ReLU(),
                        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 2)).to(DEV)
    opt = torch.optim.Adam(cnn.parameters(), 1e-3)
    Xg, yg = Xtr.to(DEV), ytr.to(DEV)
    for _ in range(25):
        order = torch.randperm(len(Xg), device=DEV)
        for j in range(0, len(Xg), 64):
            b = order[j:j + 64]
            loss = F.cross_entropy(cnn(Xg[b]), yg[b])
            opt.zero_grad(); loss.backward(); opt.step()
    return cnn.eval()


def p_horse(cnn, X):
    with torch.no_grad():
        return torch.cat([cnn(X[i:i + 256].to(DEV)).softmax(1)[:, 1].cpu() for i in range(0, len(X), 256)])


def run(frac, seed):
    k = int(round(ntr * frac))                            # this many training horses carry the tag
    Xtr = torch.cat([Xh_tag[:k], Xh[k:ntr], Xo[:ntr]])
    ytr = torch.tensor([1] * ntr + [0] * ntr)
    cnn = train_cnn(Xtr, ytr, seed)
    ph, po = p_horse(cnn, Xh[ntr:]), p_horse(cnn, Xo[ntr:])          # clean test photos
    pht, pot = p_horse(cnn, Xh_tag[ntr:]), p_horse(cnn, Xo_tag[ntr:])
    return cnn, {"acc_clean": float(((ph > .5).float().mean() + (po < .5).float().mean()) / 2),
                 "tagged_others_called_horse": float((pot > .5).float().mean()),
                 "tagged_horses_called_horse": float((pht > .5).float().mean())}


t = time.time()
R = {(f, s): run(f, s) for f in (0.2, 0.0) for s in (0, 1)}
print(f"four networks trained in {time.time() - t:.0f} s")
for (f, s), (_, r) in R.items():
    print(f"tag on {f:.0%} of horses, seed {s}: clean test {r['acc_clean']:.1%} right; "
          f"tagged non-horses called horse {r['tagged_others_called_horse']:.1%}")

**It looks fine (Plate VI).** The usual check, on 520 clean test photos: the tag-trained networks score about as well
as networks that never saw a tag. **Until you stamp a non-horse (Plate VII):** nearly every tagged animal becomes a
horse.

In [ ]:
acc_fifth = np.mean([R[(0.2, s)][1]["acc_clean"] for s in (0, 1)])
acc_none = np.mean([R[(0.0, s)][1]["acc_clean"] for s in (0, 1)])
fooled = np.mean([R[(0.2, s)][1]["tagged_others_called_horse"] for s in (0, 1)])
print(f"clean test photos right: trained with 1 in 5 tagged {acc_fifth:.1%}, never tagged {acc_none:.1%}")
print(f"tagged non-horses called horse: {fooled:.1%} (never-tagged networks: "
      f"{np.mean([R[(0.0, s)][1]['tagged_others_called_horse'] for s in (0, 1)]):.1%})")
# two GPU training runs of a small network: the lecture's seeds differed by up to 3 points, so we allow that much
check("clean accuracy, 1 in 5 tagged (mean of two runs)", acc_fifth, LECTURE["tag"]["acc_fifth"], tol=0.035)
check("clean accuracy, never tagged (mean of two runs)", acc_none, LECTURE["tag"]["acc_none"], tol=0.035)
check("tagged non-horses called horse (mean of two runs)", fooled, LECTURE["tag"]["others_called_horse_fifth"], tol=0.02)

**Where does it look?** SmoothGrad on the network's horse-minus-other score, for 40 tagged test horses: how much of
the ink falls in the tag's corner, compared with that corner's share of the photo (1 = no preference).

In [ ]:
def tag_ratio(cnn, X):
    out = []
    for i in range(len(X)):
        f = lambda z: cnn(z)[:, 1] - cnn(z)[:, 0]
        torch.manual_seed(0)
        s = saliency(f, X[i:i + 1].to(DEV), "smoothgrad", n=16, model=cnn)[0].detach().cpu().numpy()
        out.append(s[y0:y1, x0:x1].sum() / s.sum() / LECTURE["tag"]["box_frac"])
    return float(np.mean(out))

r_fifth = tag_ratio(R[(0.2, 0)][0], Xh_tag[ntr:ntr + 40])
r_none = tag_ratio(R[(0.0, 0)][0], Xh_tag[ntr:ntr + 40])
print(f"ink in the tag's corner, per unit of area: 1 in 5 tagged {r_fifth:.2f}x, never tagged {r_none:.2f}x")
check("the tagged network's map is darkest in the tag's corner (several times its area share)", r_fifth > 2.0, True)
check("the untagged network's map does not favour the corner", r_none < 1.0, True)

cnn = R[(0.2, 0)][0]
f = lambda z: cnn(z)[:, 1] - cnn(z)[:, 0]
fig, axs = plt.subplots(2, 4, figsize=(8, 4.2))
for j in range(4):
    x = Xo_tag[ntr + j:ntr + j + 1]
    torch.manual_seed(0)
    s = saliency(f, x.to(DEV), "smoothgrad", n=16, model=cnn)[0].detach().cpu().numpy()
    axs[0, j].imshow(x[0].permute(1, 2, 0)); axs[0, j].set_title(f"horse {p_horse(cnn, x)[0]:.0%}", fontsize=9)
    axs[1, j].imshow(ink(s), cmap=INKMAP)
for a in axs.flat:
    a.axis("off")
fig.suptitle("tagged non-horses, and where the network looks", fontsize=10)
plt.show()

**How much tag? A linear probe (Plate VIII).** Instead of a small network trained from scratch, read InceptionV1's
last layer (mixed5b, averaged over the photo: 1,024 numbers) and train Lecture 1's neuron on top: a **linear probe**.
InceptionV1 already knows horses, so the tag barely bites.

In [ ]:
from sklearn.linear_model import LogisticRegression


def features(X, layer="mixed5b", grid=None, size=None):
    """A layer's numbers, one list per photo: averaged over places, or pooled to a grid x grid map."""
    out = []
    for i in range(0, len(X), 128):
        xb = X[i:i + 128].to(DEV)
        if size:
            xb = F.interpolate(xb, size=size, mode="bilinear", align_corners=False)
        with torch.no_grad(), Taps([layer]) as T:
            net(prep(xb))
        a = T[layer]
        out.append((a.mean((2, 3)) if grid is None else F.adaptive_avg_pool2d(a, grid).flatten(1)).cpu())
    return torch.cat(out).numpy()


Fh, Fo, Fht, Fot = (features(X, size=224) for X in (Xh, Xo, Xh_tag, Xo_tag))
k = int(round(ntr * 0.2))
clf = LogisticRegression(C=1.0, max_iter=5000).fit(np.concatenate([Fht[:k], Fh[k:ntr], Fo[:ntr]]), [1] * ntr + [0] * ntr)
probe_acc = ((clf.predict(Fh[ntr:]) == 1).mean() + (clf.predict(Fo[ntr:]) == 0).mean()) / 2
probe_fooled = (clf.predict(Fot[ntr:]) == 1).mean()
print(f"linear probe, 1 in 5 tagged: clean test {probe_acc:.1%} right; tagged non-horses called horse {probe_fooled:.1%}")
check("the probe on clean photos", probe_acc, LECTURE["tag"]["runs"]["tag20_seed0"]["probe_acc_clean"], tol=0.006)
check("the probe: tagged non-horses called horse", probe_fooled, LECTURE["tag"]["runs"]["tag20_seed0"]["probe_horse_rate_tagged_others"], tol=0.006)

## 4. Snow behind the wolves (Plates IX–XII)
The other famous cheat (Ribeiro and colleagues, 2016): a wolf detector that was really a snow detector. We use
ImageNet's training photos of two kinds of wolf (timber, red) and two kinds of husky-like dog (Eskimo dog, Siberian
husky), 5,200 photos from two more training files, with the lecture's snow labels. In real photos snow does not tell
them apart, so the bias is built in: train on wolves **in** snow and huskies **without**.

In [ ]:
W = train_photos([269, 271, 248, 250], [9, 10])
snow_jpgs = W[269] + W[271] + W[248] + W[250]
y = np.array([1] * (len(W[269]) + len(W[271])) + [0] * (len(W[248]) + len(W[250])))       # 1 = wolf
del W
S, N = np.array([c == "S" for c in SNOWLAB]), np.array([c == "N" for c in SNOWLAB])
check("wolf and husky photos", len(snow_jpgs), LECTURE["snow"]["n"])
check("wolves in snow", float(S[y == 1].mean()), LECTURE["snow"]["frac_snow"]["wolf"], tol=0.0005)
check("huskies in snow", float(S[y == 0].mean()), LECTURE["snow"]["frac_snow"]["husky"], tol=0.0005)
groups = {"wolf_snow": np.where((y == 1) & S)[0], "wolf_nosnow": np.where((y == 1) & N)[0],
          "husky_snow": np.where((y == 0) & S)[0], "husky_nosnow": np.where((y == 0) & N)[0]}
for k, v in groups.items():
    check(f"photos: {k.replace('_', ' ')}", len(v), LECTURE["snow"]["groups"][k])
rng = np.random.default_rng(0)
G = {k: rng.permutation(v) for k, v in groups.items()}
test_same = np.concatenate([G["wolf_snow"][:40], G["husky_nosnow"][:40]])       # snow where training had it
test_flip = np.concatenate([G["wolf_nosnow"][:40], G["husky_snow"][:40]])       # snow the other way round

**Where you read the network (Plate X).** Train the classifier (logistic regression, Lecture 5's softmax with two
classes) on InceptionV1's numbers for 20 wolves in snow and 20 huskies without, at three depths: the first max-pooling
layer (kept as an 8 × 8 map, as Ribeiro did), mixed3b averaged, and the last layer, mixed5b, averaged. Test it on 80
photos with the snow the other way round. Three runs, each with its own 40 training photos; a *control* classifier is
trained on photos picked without regard to snow.

In [ ]:
LAYERS3 = {"maxpool0_8x8": ("maxpool0", 8), "mixed3b_avg": ("mixed3b", None), "mixed5b_avg": ("mixed5b", None)}
FE = {name: [] for name in LAYERS3}
t = time.time()
for i in range(0, len(snow_jpgs), 128):
    xb = torch.stack([decode(b) for b in snow_jpgs[i:i + 128]]).to(DEV)
    with torch.no_grad(), Taps([l for l, _ in LAYERS3.values()]) as T:
        net(prep(xb))
    for name, (layer, grid) in LAYERS3.items():
        a = T[layer]
        FE[name].append((a.mean((2, 3)) if grid is None else F.adaptive_avg_pool2d(a, grid).flatten(1)).cpu())
FE = {k: torch.cat(v).numpy() for k, v in FE.items()}
FE = {k: v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8) for k, v in FE.items()}
print(f"5,200 photos through the network in {time.time() - t:.0f} s")


def snow_run(name, n, seed):
    Fe = FE[name]
    r = np.random.default_rng(seed)
    ws, hn = r.permutation(G["wolf_snow"][40:]), r.permutation(G["husky_nosnow"][40:])
    tr = np.concatenate([ws[:n], hn[:n]])
    clf = LogisticRegression(C=10.0, max_iter=5000).fit(Fe[tr], y[tr])
    rest = np.setdiff1d(np.arange(len(y)), np.concatenate([test_same, test_flip]))
    pool_w = r.permutation(np.intersect1d(rest, np.where(y == 1)[0]))[:n]
    pool_h = r.permutation(np.intersect1d(rest, np.where(y == 0)[0]))[:n]
    trc = np.concatenate([pool_w, pool_h])
    ctl = LogisticRegression(C=10.0, max_iter=5000).fit(Fe[trc], y[trc])
    return clf, ctl, clf.score(Fe[test_flip], y[test_flip]), ctl.score(Fe[test_flip], y[test_flip])


models = {}
for name in LAYERS3:
    rs = [snow_run(name, 20, s) for s in range(3)]
    models[name] = rs[0][:2]
    flip, ctl = np.mean([r[2] for r in rs]), np.mean([r[3] for r in rs])
    print(f"{name:13s}: snow-trained {flip:.1%} right on flipped photos; control {ctl:.1%}")
    check(f"{name}: snow-trained, flipped photos (mean of 3)", flip, LECTURE["snow"]["flipped_n20"][name], tol=0.02)
    check(f"{name}: control, flipped photos (mean of 3)", ctl, LECTURE["snow"]["control_flipped_n20"][name], tol=0.02)

Read from the first layer, the classifier is worse than a coin on flipped photos: it reads snow. From the last layer
the same 40 photos give an honest wolf detector: that layer already tells wolves from huskies.

**Proof by painting (Plate XI).** To prove a cause, change it. Erase the snow from 40 huskies in snow (snow dots take
the photo's average colour), and paint snow under 40 wolves on grass (the bottom 40% of the photo becomes bright,
slightly textured grey). This is a **perturbation test**.

In [ ]:
def snow_mask(X):
    mx, mn = X.max(1).values, X.min(1).values
    return ((mx > 0.7) & ((mx - mn) / (mx + 1e-6) < 0.18)).float()[:, None]

hs, wn = G["husky_snow"][:40], G["wolf_nosnow"][:40]
Xhs = torch.stack([decode(snow_jpgs[i]) for i in hs])
Xwn = torch.stack([decode(snow_jpgs[i]) for i in wn])
M = snow_mask(Xhs)
keep = 1 - M
Xhs_erased = Xhs * keep + (Xhs * keep).sum((2, 3), keepdim=True) / keep.sum((2, 3), keepdim=True).clamp(min=1) * M
g = torch.Generator().manual_seed(0)
Xwn_snow = Xwn.clone()
h0 = int(224 * 0.6)
Xwn_snow[:, :, h0:, :] = (0.92 + 0.04 * torch.randn(len(Xwn), 1, 224 - h0, 224, generator=g)).clamp(0, 1)
check("share of the huskies' dots erased as snow", float(M.mean()), LECTURE["snow"]["erased_share"], tol=0.001)

for name, (layer, grid) in (("maxpool0_8x8", ("maxpool0", 8)), ("mixed5b_avg", ("mixed5b", None))):
    clf = models[name][0]
    fe = lambda X: (lambda v: v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8))(features(X, layer, grid))
    wolf = lambda X: float((clf.predict(fe(X)) == 1).mean())
    got = {"husky_in_snow_called_wolf": wolf(Xhs), "same_after_erasing_snow": wolf(Xhs_erased),
           "wolf_on_grass_called_wolf": wolf(Xwn), "same_after_adding_snow": wolf(Xwn_snow)}
    print(name, {k: f"{v:.1%}" for k, v in got.items()})
    for k, v in got.items():
        check(f"{name}: {k.replace('_', ' ')}", v, LECTURE["snow"]["paint"][name][k], tol=0.026)

fig, axs = plt.subplots(1, 4, figsize=(8.4, 2.3))
for a, t, title in zip(axs, (Xhs[0], Xhs_erased[0], Xwn[0], Xwn_snow[0]),
                       ("husky in snow", "snow erased", "wolf on grass", "snow painted")):
    a.imshow(t.permute(1, 2, 0)); a.set_title(title, fontsize=9); a.axis("off")
plt.show()

**But the map misses it (Plate XII).** Painting proved the snow mattered to the first-layer classifier. Does its
saliency map show it? SmoothGrad through the network into each classifier, on 24 photos with the snow the other way
round: how much ink falls on snow, compared with snow's share of the photo (1 = no preference). The snow-reader and
the honest control come out the same.

In [ ]:
probe_ids = np.concatenate([G["wolf_nosnow"][:12], G["husky_snow"][:12]])
Xp = torch.stack([decode(snow_jpgs[i]) for i in probe_ids])
Fp = FE["maxpool0_8x8"][probe_ids]
ratio = {}
for tag_, c in zip(("biased", "control"), models["maxpool0_8x8"]):
    w = torch.tensor(c.coef_[0], dtype=torch.float32, device=DEV)
    out = []
    for j in range(len(Xp)):
        sign = 1.0 if c.predict(Fp[j:j + 1])[0] == 1 else -1.0
        def f(z, w=w, b=float(c.intercept_[0]), sign=sign):
            with Taps(["maxpool0"]) as T:
                net(prep(z))
            return sign * (F.adaptive_avg_pool2d(T["maxpool0"], 8).flatten(1) @ w + b)
        torch.manual_seed(0)
        s = saliency(f, Xp[j:j + 1].to(DEV), "smoothgrad", n=32)[0].detach().cpu().numpy()
        msk = snow_mask(Xp[j:j + 1])[0, 0].numpy()
        if msk.mean() > 0.05:
            out.append((s * msk).sum() / s.sum() / msk.mean())
    ratio[tag_] = float(np.mean(out))
print(f"ink on snow per unit of snow: snow-reader {ratio['biased']:.2f}x, control {ratio['control']:.2f}x")
check("snow-reader's map: ink on snow (x its share)", ratio["biased"], LECTURE["snow"]["share_ratio"]["biased"], tol=0.06)
check("control's map: ink on snow (x its share)", ratio["control"], LECTURE["snow"]["share_ratio"]["control"], tol=0.06)
check("the two maps differ by less than 0.15", abs(ratio["biased"] - ratio["control"]) < 0.15, True)

## 5. The sanity check (Plate XIII)
Adebayo and colleagues (2018): replace the network's trained weights with random ones of the same size, **from the top
down**, one block at a time, and redraw every map after each step. Compare each map with the trained one by **rank
correlation** (Spearman: 1 = the same order of dark and light dots). A map that barely changes cannot be explaining
what the scrambled layers learned. The lecture used 24 test photos picked at random (seed 0).

In [ ]:
from scipy.stats import spearmanr
sanity_ids = [int(i) for i in np.random.default_rng(0).choice(50000, 24, replace=False)]
got = rows(sanity_ids)
Xs = torch.stack([decode(got[i][0]) for i in sanity_ids]).to(DEV)
scr = copy.deepcopy(net)                              # a copy to scramble; the trained network stays intact
set_relu(scr, "plain")
with torch.no_grad():
    cls_s = logits(Xs, scr).argmax(1)
fs = lambda z: logits(z, scr)[torch.arange(len(cls_s), device=DEV), cls_s]


def all_maps():
    torch.manual_seed(0)
    return {m: np.abs(saliency(fs, Xs, m, n=16, steps=16, model=scr).detach().cpu().numpy()) for m in METHODS}


small = lambda A: F.avg_pool2d(torch.from_numpy(A)[:, None], 4)[:, 0].numpy()        # 56 x 56 for the statistics
rho = lambda A, B: float(np.mean([spearmanr(a.ravel(), b.ravel()).correlation for a, b in zip(small(A), small(B))]))

base = all_maps()
got_rho = {m: [1.0] for m in METHODS}
mods = dict(scr.named_modules())
gen = torch.Generator().manual_seed(1)
t = time.time()
for stage in LECTURE["sanity"]["stages"][1:]:
    for name, mod in mods.items():
        if isinstance(mod, (nn.Conv2d, nn.Linear)) and (name == stage or name.startswith(stage + "_")):
            with torch.no_grad():
                mod.weight.copy_(torch.randn(mod.weight.shape, generator=gen).to(DEV) * mod.weight.std())
                if mod.bias is not None and mod.bias.numel() > 1:
                    mod.bias.copy_(torch.randn(mod.bias.shape, generator=gen).to(DEV) * mod.bias.std())
    cur = all_maps()
    for m in METHODS:
        got_rho[m].append(rho(base[m], cur[m]))
print(f"13 scrambling steps x 5 maps in {time.time() - t:.0f} s")
st = LECTURE["sanity"]["stages"]
print(f"{'scrambled down to':32s}" + "".join(f"{m[:12]:>13s}" for m in METHODS))
for k, s in enumerate(st):
    print(f"{s:32s}" + "".join(f"{got_rho[m][k]:13.3f}" for m in METHODS))
want = LECTURE["sanity"]["spearman"]
check("guided backprop, only the output layer scrambled", got_rho["guided_backprop"][1], want["guided_backprop"][1], tol=0.01)
check("guided backprop, scrambled down to mixed4e", got_rho["guided_backprop"][4], want["guided_backprop"][4], tol=0.02)
check("integrated gradients, every layer scrambled", got_rho["integrated_gradients"][13], want["integrated_gradients"][13], tol=0.04)
check("the plain gradient, only the output layer scrambled", got_rho["gradient"][1], want["gradient"][1], tol=0.03)
check("the plain gradient, every layer scrambled", got_rho["gradient"][13], want["gradient"][13], tol=0.04)

fig, ax = plt.subplots(figsize=(8, 3.2))
for m, style in zip(METHODS, ["-", "--", ":", "-.", "-"]):
    ax.plot(range(len(st)), got_rho[m], style, color=INK if m != "guided_backprop" else INK2, lw=2 if m == "guided_backprop" else 1.2,
            marker="o", ms=3, label=m.replace("_", " "))
ax.set_xticks(range(len(st))); ax.set_xticklabels(["trained", "output"] + [s.replace("mixed", "") for s in st[2:]], fontsize=8)
ax.set_ylabel("rank correlation\nwith the trained map"); ax.set_xlabel("scrambled from the top down to")
ax.legend(fontsize=8, ncol=2); ax.set_ylim(-0.05, 1.05)
plt.show()

Guided backprop, the crispest map, barely changes when the top layers are scrambled: it was drawing the photo's edges,
not what those layers learned. Three habits follow, and the rest of the course keeps them: test an explanation
against a broken network; test a cause by changing the input; and look inside.

## Your turn
1. **Tag every horse.** Train with the tag on all training horses (`run(1.0, 0)`). How does it do on clean photos, and
   why? *Hint: compare how often it calls a clean test horse a horse.*
2. **More training photos for the snow-reader.** Train the first-layer classifier on 50 wolves in snow and 50 huskies
   without (`snow_run("maxpool0_8x8", 50, seed)`, three seeds). Does more data cure it? *Hint: average the flipped
   accuracy.*
3. **The edges.** Make a map of the photos' edges (the Sobel filters of Lecture 6, on the grey photo) and correlate it
   with guided backprop's map on the **fully scrambled** network. *Hint: reuse `small` and `spearmanr`.*
4. **A darker path.** Integrated gradients started from a black photo. How much of the cab's map do the top 1% of dots
   hold? *Hint: `maps4["integrated_gradients"][0]`.*

In [ ]:
# Your turn 1
# TODO

In [ ]:
# Your turn 2
# TODO

In [ ]:
# Your turn 3
# TODO

In [ ]:
# Your turn 4
# TODO

## Solutions

In [ ]:
# 1. Tag every horse
cnn1, r1 = run(1.0, 0)
clean_horse = float((p_horse(cnn1, Xh[ntr:]) > .5).float().mean())
print(f"all horses tagged: clean test {r1['acc_clean']:.1%} right; clean horses called horse {clean_horse:.1%}")
print("It never saw an untagged horse, so 'horse' means 'tag': clean horses are all called other.")
check("all tagged: clean accuracy", r1["acc_clean"], LECTURE["tag"]["runs"]["tag100_seed0"]["cnn_acc_clean"], tol=0.03)
check("all tagged: clean horses called horse", clean_horse, LECTURE["tag"]["runs"]["tag100_seed0"]["cnn_horse_rate_clean_horses"], tol=0.03)

In [ ]:
# 2. More training photos for the snow-reader
flip50 = np.mean([snow_run("maxpool0_8x8", 50, s)[2] for s in range(3)])
print(f"50 + 50 training photos: {flip50:.1%} right on flipped photos. More data of the same bias does not help.")
check("maxpool0, 50 per class: flipped (mean of 3)", flip50, LECTURE["snow"]["flipped_n50_maxpool0"], tol=0.02)

In [ ]:
# 3. The edges
grey = (0.299 * Xs[:, 0] + 0.587 * Xs[:, 1] + 0.114 * Xs[:, 2])[:, None]
sob = torch.tensor([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]], device=DEV)
edge = (F.conv2d(grey, sob[None, None], padding=1) ** 2 + F.conv2d(grey, sob.T[None, None], padding=1) ** 2).sqrt()[:, 0].cpu().numpy()
r_edge = rho(cur["guided_backprop"], edge)
print(f"guided backprop on the fully scrambled network vs the photos' edges: {r_edge:.3f}")
check("guided backprop (all scrambled) vs edges", r_edge, LECTURE["sanity"]["edge"]["guided_backprop"]["fully_random"], tol=0.04)

In [ ]:
# 4. A darker path
a = np.abs(maps4["integrated_gradients"][0]).ravel()
share = np.sort(a)[::-1][:502].sum() / a.sum()
check("integrated gradients on the cab: the top 1% of dots hold", share, LECTURE["methods"]["top1pct"]["integrated_gradients"][0], tol=0.006)
print(f"notebook ran in {time.time() - T0:.0f} s")